In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random, re, torch
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from msnap.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from msnap.utils.container_utils import chunks
from msnap.core import msnap_prompts

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'

# zero_shot_target = 'fact'
zero_shot_target = 'other'

in_dir = f'{data_dir}/check_targets/zero_shot_{zero_shot_target}'
out_dir = f'{data_dir}/save_hiddens/zero_shot_{zero_shot_target}'

In [ ]:
model_name = 'Llama-3.2-3B'
# model_name = 'Llama-3.1-8B'

model_name_or_path = f'meta-llama/{model_name}-Instruct'
dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

model: AutoModelForCausalLM = model_utils.get_model(model_name_or_path, dtype, device, is_eval=True)

# 평가/추론 시에는 반드시 'left' 패딩
tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

In [ ]:
def make_prompts_contrastive_activation(datas: list, trigger='Answer: '):
    prompts_fact, prompts_counter = [], []

    for data in datas:
        prompt = data['prompt']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']

        chat_prompt = tokenizer.apply_chat_template(
            prompt,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts_fact.append(chat_prompt + trigger + answer_fact)
        prompts_counter.append(chat_prompt + trigger + answer_counter)
    
    return prompts_fact, prompts_counter

In [ ]:
def save_hiddens(prompts, batch_size, out_prefix):
    num_layers = model.config.num_hidden_layers
    print(f'save_hiddens() num_layers : {num_layers}\n')

    # outputs.hidden_states는 (초기 임베딩 + 각 레이어 출력)이므로 길이는 num_layers + 1
    h_states_accumulator = {i: [] for i in range(num_layers+1)}

    batch_idx = 0
    for prompts_batch in chunks(prompts, batch_size):
        inputs = tokenizer(
            prompts_batch,
            padding=True,
            truncation=True,
            max_length=max_seq_length,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        for layer_idx, h_states in enumerate(outputs.hidden_states):
            # 마지막 토큰 추출 -> CPU로 이동 -> float32 캐스팅
            last_token_h_state = h_states[:, -1, :].cpu().to(torch.float32)
            h_states_accumulator[layer_idx].append(last_token_h_state)
        
        batch_idx += 1
        if (batch_idx*batch_size) % 1000 == 0:
            print(f'save_hiddens() {(batch_idx*batch_size)} forward complet.')


    # 리스트에 모인 배치 텐서들을 하나의 큰 텐서로 병합 후 파일로 저장
    for layer_idx in range(num_layers+1):
        layer_h_states = torch.cat(h_states_accumulator[layer_idx], dim=0)
        print(f'save_hiddens() layer_{layer_idx}_h_states shape : {layer_h_states.shape}')

        out_file_path = f'{out_prefix}/layer_{layer_idx}_h_states.pt'
        file_utils.make_parent(out_file_path)
        torch.save(layer_h_states, out_file_path)
        # print(f'layer_{layer_idx} saved -> {out_file_path}\n')
    print()

In [ ]:
in_file_path_temp = f'{in_dir}/{model_name}/maked/msnap_gpt-5.2_maked_targets_{model_name}' + '_F-{}_C-{}.json'

fact_sizes = [3]
counter_poss = ['shuffle']

# fact_sizes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
# counter_poss = ['front', 'back', 'shuffle']

for fact_size in fact_sizes:
    for counter_pos in counter_poss:
        in_file_path = in_file_path_temp.format(fact_size, counter_pos)
        datas = json_utils.load_json(in_file_path)

        prompts_fact, prompts_counter = make_prompts_contrastive_activation(datas)
        print(f'fact_size : {fact_size}, counter_pos : {counter_pos}')

        print(f'prompts_fact size : {len(prompts_fact)}')
        out_prefix = f'{out_dir}/{model_name}/fact'
        save_hiddens(prompts_fact, 20, out_prefix)

        print(f'prompts_counter size : {len(prompts_counter)}')
        out_prefix = f'{out_dir}/{model_name}/counter'
        save_hiddens(prompts_counter, 20, out_prefix)
        
    #     break
    # break